# noScat4: old model vs new (ggd_mass) scalar potentials
Same run set-up (M=256, J=8, Q=1+3, seed 900), no Scat4. Run on Jean Zay (needs `saved_results/`). Built on `run_comparison.py`.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
from run_comparison import *
pd.set_option('display.float_format', '{:.2f}'.format, 'display.width', 250, 'display.max_rows', 100)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
RUNS = {   # {display name: --label}; order fixes the colours
    'old (final_noScat4)': 'turbulence_final_noScat4',
    'new (ggd_mass_noScat4)': 'turbulence_ggd_mass_noScat4',
}
SEED, THRESHOLD = 900, 1e-8
runs, missing = load_runs(RUNS, seed=SEED)
assert not missing, missing
cfg0 = next(iter(runs.values()))['cfg']
x1 = reference_data(cfg0)
xs = {k: r['xt'] for k, r in runs.items()}
print({k: tuple(v.shape) for k, v in xs.items()}, tuple(x1.shape))

## Per-channel table: generated / data (1 = perfect)
m{p} = E|W|^p ratio, q{a} = quantile of |W| ratio. Row j0q0 = finest. `n_series=10000` uses all series (as `compute_marginal_agreement.py`). Green = run closest to 1, red = farthest (no colour if they differ by < 1%).

In [ ]:
N = len(x1)
for bank, J in (('morlet', 8), ('psi', 8)):
    print(f'Q={BANKS[bank]}')
    display(style_channel_table(channel_table(x1, xs, bank=bank, J=J, n_series=N)))

## Tails per channel (Q=1 then Q=3): histogram and survival function

In [ ]:
plot_tail_hists(x1, xs, bank='morlet', n_ch=8, n_series=N)

In [ ]:
plot_tail_hists(x1, xs, bank='psi', n_ch=9, n_series=N)

## Statistics
Physical statistics, then the imposed moments at the end of the run (the two runs have different potential sets, so only the error distributions are comparable).

In [ ]:
S_ref = compute_stats(x1)
S_runs = {k: compute_stats(v) for k, v in xs.items()}
ks = wavelet_ks(S_ref, S_runs)
summary_table(S_ref, S_runs, runs, THRESHOLD, ks=ks)

In [ ]:
plot_marginals(S_ref, S_runs); plot_spectra(S_ref, S_runs); plot_structure(S_ref, S_runs); plot_structure_orders(S_ref, S_runs); plot_increment_pdfs(S_ref, S_runs)

In [ ]:
plot_moment_matching(runs, THRESHOLD)